In [1]:
# ============================================================
# Portfolio Performance Analytics Dashboard
# Data Collection
# ============================================================

# Import required libraries
import pandas as pd
import yfinance as yf

In [2]:
# ------------------------------------------------------------
# Define stock tickers and date range
# ------------------------------------------------------------

tickers = ["MSFT", "ASML", "NVDA", "AMZN", "GOOGL"]

start_date = "2023-01-01"
end_date = "2026-01-01"

In [3]:
# ------------------------------------------------------------
# Download daily stock data from Yahoo Finance
# ------------------------------------------------------------

print("Downloading stock market data...")

raw_data = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    group_by="ticker",
    auto_adjust=True
)

print("Download complete.")

[*********************100%***********************]  5 of 5 completed

Download complete.


In [4]:
# ------------------------------------------------------------
# View the structure of the downloaded dataset
# ------------------------------------------------------------

print(raw_data.head())

print("\nDataset Shape:")
print(raw_data.shape)

Ticker           NVDA                                                  GOOGL  \
Price            Open       High        Low      Close     Volume       Open   
Date                                                                           
2023-01-03  14.818076  14.962755  14.064750  14.283264  401277000  88.855599   
2023-01-04  14.534706  14.820071  14.209429  14.716302  431324000  89.609375   
2023-01-05  14.458875  14.531713  14.116635  14.233377  389168000  86.752987   
2023-01-06  14.441912  14.976724  14.002888  14.826058  405044000  86.078559   
2023-01-09  15.250114  16.020402  15.107431  15.593351  504231000  87.635679   

Ticker                                                 ...        MSFT  \
Price            High        Low      Close    Volume  ...        Open   
Date                                                   ...               
2023-01-03  90.303637  87.794370  88.389458  28131200  ...  236.351375   
2023-01-04  89.906919  86.554621  87.357986  34854800  ...  225

In [5]:
# ------------------------------------------------------------
# Create a clean dataset for all stocks
# ------------------------------------------------------------

all_stocks = []

for ticker in tickers:

    # Extract data for one ticker
    stock_df = raw_data[ticker].copy()

    # Reset index so Date becomes a column
    stock_df = stock_df.reset_index()

    # Add ticker symbol
    stock_df["Ticker"] = ticker

    # Keep only required fields
    stock_df = stock_df[
        [
            "Date",
            "Ticker",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ]

    all_stocks.append(stock_df)

# Combine all stocks into one dataframe
portfolio_df = pd.concat(all_stocks, ignore_index=True)

print("Combined dataset created.")

Combined dataset created.


In [6]:
# ------------------------------------------------------------
# Verify dataset structure
# ------------------------------------------------------------

print(portfolio_df.head())

print("\nNumber of records:")
print(len(portfolio_df))

print("\nStocks included:")
print(portfolio_df["Ticker"].unique())

Price       Date Ticker        Open        High         Low       Close  \
0     2023-01-03   MSFT  236.351375  238.947466  230.828594  232.948257   
1     2023-01-04   MSFT  225.850299  226.423964  219.705250  222.758331   
2     2023-01-05   MSFT  220.910952  221.251270  215.621532  216.156311   
3     2023-01-06   MSFT  216.827212  219.510808  213.278252  218.703781   
4     2023-01-09   MSFT  220.181678  224.839095  220.142792  220.833130   

Price    Volume  
0      25740000  
1      50623400  
2      39585600  
3      43613600  
4      27369800  

Number of records:
3760

Stocks included:
['MSFT' 'ASML' 'NVDA' 'AMZN' 'GOOGL']


In [7]:
# ------------------------------------------------------------
# Data quality check
# ------------------------------------------------------------

print("Missing values by column:")

print(portfolio_df.isnull().sum())

Missing values by column:
Price
Date      0
Ticker    0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


In [8]:
# ------------------------------------------------------------
# Save dataset for future use
# ------------------------------------------------------------

portfolio_df.to_csv(
    "portfolio_stock_data.csv",
    index=False
)

print("File saved: portfolio_stock_data.csv")

File saved: portfolio_stock_data.csv


In [9]:
# ------------------------------------------------------------
# Clean column names if Yahoo Finance created a MultiIndex
# ------------------------------------------------------------

portfolio_df.columns = [col[0] if isinstance(col, tuple) else col
                        for col in portfolio_df.columns]

print(portfolio_df.columns)

Index(['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')


In [10]:
# ------------------------------------------------------------
# Sort data by ticker and date
# ------------------------------------------------------------

portfolio_df = portfolio_df.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100


In [11]:
# ------------------------------------------------------------
# Calculate Daily Returns
# ------------------------------------------------------------

portfolio_df["Daily_Return"] = (
    portfolio_df.groupby("Ticker")["Close"]
    .pct_change()
)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870


In [12]:
# ------------------------------------------------------------
# Calculate Cumulative Returns
# ------------------------------------------------------------

portfolio_df["Cumulative_Return"] = (
    portfolio_df.groupby("Ticker")["Daily_Return"]
    .transform(lambda x: (1 + x).cumprod() - 1)
)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945


In [13]:
# ------------------------------------------------------------
# Calculate 50-Day and 200-Day Moving Averages
# ------------------------------------------------------------

portfolio_df["SMA_50"] = (
    portfolio_df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(50).mean())
)

portfolio_df["SMA_200"] = (
    portfolio_df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(200).mean())
)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN


In [14]:
# ------------------------------------------------------------
# Calculate 21-Day Rolling Volatility
# ------------------------------------------------------------

portfolio_df["Volatility_21D"] = (
    portfolio_df.groupby("Ticker")["Daily_Return"]
    .transform(lambda x: x.rolling(21).std())
)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200,Volatility_21D
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN,NaN
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN,NaN
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN,NaN
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN,NaN
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN,NaN


In [15]:
# ------------------------------------------------------------
# Calculate Running Peak
# ------------------------------------------------------------

portfolio_df["Running_Peak"] = (
    portfolio_df.groupby("Ticker")["Close"]
    .cummax()
)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200,Volatility_21D,Running_Peak
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN,NaN,85.820000
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN,NaN,85.820000
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN,NaN,85.820000
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN,NaN,86.080002
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN,NaN,87.360001


In [16]:
# ------------------------------------------------------------
# Calculate Drawdown Percentage
# ------------------------------------------------------------

portfolio_df["Drawdown_Pct"] = (
    (
        portfolio_df["Close"]
        - portfolio_df["Running_Peak"]
    )
    / portfolio_df["Running_Peak"]
) * 100

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200,Volatility_21D,Running_Peak,Drawdown_Pct
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN,NaN,85.820000,0.000000
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN,NaN,85.820000,-0.792356
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN,NaN,85.820000,-3.146116
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN,NaN,86.080002,0.000000
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN,NaN,87.360001,0.000000


In [17]:
# ------------------------------------------------------------
# Calculate 50-Day Average Volume
# ------------------------------------------------------------

portfolio_df["Avg_Volume_50"] = (
    portfolio_df.groupby("Ticker")["Volume"]
    .transform(lambda x: x.rolling(50).mean())
)

In [18]:
# ------------------------------------------------------------
# Create Volume Breakout Flag
# ------------------------------------------------------------

portfolio_df["Is_Breakout_Day"] = (
    portfolio_df["Volume"]
    > portfolio_df["Avg_Volume_50"]
)

portfolio_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200,Volatility_21D,Running_Peak,Drawdown_Pct,Avg_Volume_50,Is_Breakout_Day
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN,NaN,85.820000,0.000000,NaN,False
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN,NaN,85.820000,-0.792356,NaN,False
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN,NaN,85.820000,-3.146116,NaN,False
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN,NaN,86.080002,0.000000,NaN,False
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN,NaN,87.360001,0.000000,NaN,False


In [19]:
# ------------------------------------------------------------
# Verify missing values
# ------------------------------------------------------------

portfolio_df.isnull().sum()

Date                   0
Ticker                 0
Open                   0
High                   0
Low                    0
Close                  0
Volume                 0
Daily_Return           5
Cumulative_Return      5
SMA_50               245
SMA_200              995
Volatility_21D       105
Running_Peak           0
Drawdown_Pct           0
Avg_Volume_50        245
Is_Breakout_Day        0
dtype: int64

In [20]:
# ------------------------------------------------------------
# Review engineered dataset
# ------------------------------------------------------------

portfolio_df.head(10)

,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200,Volatility_21D,Running_Peak,Drawdown_Pct,Avg_Volume_50,Is_Breakout_Day
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN,NaN,85.820000,0.000000,NaN,False
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN,NaN,85.820000,-0.792356,NaN,False
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN,NaN,85.820000,-3.146116,NaN,False
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN,NaN,86.080002,0.000000,NaN,False
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN,NaN,87.360001,0.000000,NaN,False
5,2023-01-10,AMZN,87.570000,90.190002,87.290001,89.870003,67756600,0.028732,0.047192,NaN,NaN,NaN,89.870003,0.000000,NaN,False
6,2023-01-11,AMZN,90.930000,95.260002,90.930000,95.089996,103126200,0.058084,0.108017,NaN,NaN,NaN,95.089996,0.000000,NaN,False
7,2023-01-12,AMZN,96.930000,97.190002,93.500000,95.269997,85254800,0.001893,0.110114,NaN,NaN,NaN,95.269997,0.000000,NaN,False
8,2023-01-13,AMZN,94.180000,98.370003,94.120003,98.120003,85549400,0.029915,0.143323,NaN,NaN,NaN,98.120003,0.000000,NaN,False
9,2023-01-17,AMZN,98.680000,98.889999,95.730003,96.050003,72755000,-0.021097,0.119203,NaN,NaN,NaN,98.120003,-2.109661,NaN,False


In [21]:
# ------------------------------------------------------------
# Save prepared dataset
# ------------------------------------------------------------

portfolio_df.to_csv(
    "portfolio_stock_metrics.csv",
    index=False
)

print("portfolio_stock_metrics.csv saved successfully.")

portfolio_stock_metrics.csv saved successfully.


In [22]:
# Quick verification

check_df = pd.read_csv("portfolio_stock_metrics.csv")

print(check_df.shape)
check_df.head()

(3760, 16)


,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,Cumulative_Return,SMA_50,SMA_200,Volatility_21D,Running_Peak,Drawdown_Pct,Avg_Volume_50,Is_Breakout_Day
0,2023-01-03,AMZN,85.459999,86.959999,84.209999,85.820000,76706000,NaN,NaN,NaN,NaN,NaN,85.820000,0.000000,NaN,False
1,2023-01-04,AMZN,86.550003,86.980003,83.360001,85.139999,68885100,-0.007924,-0.007924,NaN,NaN,NaN,85.820000,-0.792356,NaN,False
2,2023-01-05,AMZN,85.330002,85.419998,83.070000,83.120003,67930800,-0.023726,-0.031461,NaN,NaN,NaN,85.820000,-3.146116,NaN,False
3,2023-01-06,AMZN,83.029999,86.400002,81.430000,86.080002,83303400,0.035611,0.003030,NaN,NaN,NaN,86.080002,0.000000,NaN,False
4,2023-01-09,AMZN,87.459999,89.480003,87.080002,87.360001,65266100,0.014870,0.017945,NaN,NaN,NaN,87.360001,0.000000,NaN,False


In [23]:
portfolio_df.columns

Index(['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
       'Daily_Return', 'Cumulative_Return', 'SMA_50', 'SMA_200',
       'Volatility_21D', 'Running_Peak', 'Drawdown_Pct', 'Avg_Volume_50',
       'Is_Breakout_Day'],
      dtype='object')